### Importing Libraries

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import pandas as pd
import pickle
from config import ROOT_DIR
from src.models.baseline_model import create_baseline_recommender,create_recency_baseline
from src.models.als_model import train_and_save_als_model, create_als_recommender
from src.evaluation.metrics import evaluate_all, evaluate_candidates
from src.candidate_retrieval.PreviousPurchases import PreviousPurchases
from src.candidate_retrieval.ItemSimilarity import ItemSimilarityCandidates
from src.candidate_retrieval.RecencyCandidates import RecencyCandidates
from src.candidate_retrieval.AlsCandidates import ALSCandidates
from src.candidate_retrieval.StyleAlternatives import SameProductColorCandidates
from src.candidate_retrieval.CandidateRetrievalPipeline import CandidateGeneratorEnsemble
from src.utils.generate_labels import create_labels
from src.feature_engineer.feature_generator import FeatureGenerator
from src.utils.downsample_candidates import downsample_negatives
from src.models.rank_model import train_ranker, create_lgbm_recommender
from pathlib import Path

In [2]:
cleaned_df_folder = "input/cleaned"
def load_from_parquet(cleaned_df_name):
    print("Loading data from Parquet...")
    return pd.read_parquet(cleaned_df_folder + "/" + cleaned_df_name + ".parquet")

In [3]:
cols_needed = ['t_dat', 'customer_id', 'article_id','week']

train_transactions = pd.read_parquet(
    "input/cleaned/train_transactions.parquet", 
    columns=cols_needed
)

In [4]:

val_transactions = load_from_parquet("val_transactions")
articles = load_from_parquet("articles")
customers = load_from_parquet("customers")

Loading data from Parquet...
Loading data from Parquet...
Loading data from Parquet...


In [ ]:
train_target_week = 1
train_target_tx = train_transactions[train_transactions['week'] == train_target_week] # week 1
train_history_tx = train_transactions[train_transactions['week'] > train_target_week] # Weeks 2 to 104

### Baseline model

In [118]:
# Initialize the baseline recommender to get the top-k recommendations.
my_baseline_model = create_baseline_recommender(train_history_tx)

# Evaluate the baseline model on the validation transactions.
evaluate_all(my_baseline_model, train_target_tx)

Generating predictions for 72,019 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.0034440087457057516,
 'HitRate@12': 0.02477124092253433,
 'NDCG@12': 0.006318058900555214,
 'Recall@12': 0.008970650289728576}

In [6]:
# Evaluate the recency baseline model on the training target transactions.
recency_model = create_recency_baseline(train_history_tx)
evaluate_all(recency_model, train_target_tx)


Generating predictions for 72,019 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.005770575041379203,
 'HitRate@12': 0.055485358030519724,
 'NDCG@12': 0.012033335990458138,
 'Recall@12': 0.02042971796233196}

### Collaborative filtering using ALS modelling

The training dataset is massive with around 1.3M unique customers and around 100k unique articles. This makes the collaborative filtering model to run very slowly on the entire dataset as the sparse matrix is massive.

So, down sampling only the last 2 months records would help in running the ALS model faster and also the data would be more relative to recommend as the customer's fashion sense from last 2 years would not matter much.

In [7]:
train_history_tx["week"].min()

2

In [ ]:
als_train_history_file_path = ROOT_DIR / 'saved_models' / 'als_recommender_model_training_history.pkl'
als_val_history_file_path = ROOT_DIR / 'saved_models' / 'als_recommender_model_val_history.pkl'

# Train als model for train_history
train_and_save_als_model(
    train=train_history_tx,
    model_path=als_train_history_file_path,
    lookback_weeks=14,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

ALS model saved to C:\Users\yeshw\Documents\Gayathri personal doc\Gayathri\arizona\INFO 556 - Text Retrieval and Web Search\project\hm-fashion-recommendation\saved_models\als_model.pkl


{'name': 'als_model'}

In [ ]:
# Load the ALS model and mappings 
with open(als_train_history_file_path, "rb") as f:
    artifacts = pickle.load(f)
    
als_recommender = create_als_recommender(
    als_model=artifacts["model"],
    user_map=artifacts["user_map"],
    item_map=artifacts["item_map"],
    reverse_item_map=artifacts["reverse_item_map"],
    user_item_matrix=artifacts["user_item_matrix"],
    baseline=recency_model  # Fallback for cold-start or unknown customers
)
evaluate_all(als_recommender, train_target_tx)

Generating predictions for 72,019 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.022658440841382056,
 'HitRate@12': 0.10226468015384829,
 'NDCG@12': 0.03471354228484876,
 'Recall@12': 0.050577573405083315}

### Candidate Retrieval


In [61]:
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 20}),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
prev_recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(prev_recs_df):,} total candidate rows.")
prev_recs_df.head()

Generating candidates via ensemble for 68,984 validation customers...
Generated 418,740 total candidate rows.


,customer_id,article_id
42131,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0905365002
202526,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0876411001
133901,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0707269007
129279,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0934054002
101169,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0865076006


In [71]:

val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 20}),
    "item_cooccurrence": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 30
        },
    ),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
sim_recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(sim_recs_df):,} total candidate rows.")
sim_recs_df.head()

Generating candidates via ensemble for 68,984 validation customers...
Generated candidate scores for users
Generated 1,474,109 total candidate rows.


,customer_id,article_id,source
1474108,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0898694001,item_cooccurrence
165482,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0895610005,previous_purchases
1474081,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0751471001,item_cooccurrence
1474080,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0910601002,item_cooccurrence
1474079,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0905365001,item_cooccurrence


In [116]:
evaluate_candidates(recs_df, val_transactions, k=50)

Preparing evaluation matrices...
Recall@50: 0.0668


0.06678938681997981

In [43]:
# 2. Count how many candidates each customer received
counts_per_user = recs_df.groupby('customer_id').size()

# 3. Group by candidate count to see the distribution
distribution = counts_per_user.value_counts().reset_index()
distribution.columns = ['num_candidates', 'num_customers']
distribution = distribution.sort_values('num_candidates',ascending=False)

# Print the binned breakdown
print(distribution.to_string(index=False))

# Also check how many validation customers got 0 candidates entirely
total_val_customers = len(val_customers)
customers_with_recs = counts_per_user.shape[0]
print(f"\nTotal Validation Customers: {total_val_customers:,}")
print(f"Customers with at least 1 candidate: {customers_with_recs:,}")
print(f"Customers with 0 candidates (Dropped/Cold Start): {total_val_customers - customers_with_recs:,}")

 num_candidates  num_customers
            100              4
             99              8
             98             15
             97             49
             96             74
             95            118
             94            146
             93            204
             92            195
             91            331
             90            374
             89            481
             88            566
             87            683
             86            766
             85            893
             84           1120
             83           1209
             82           1380
             81           1456
             80           1471
             79           1629
             78           1706
             77           1684
             76           1779
             75           1899
             74           1991
             73           1927
             72           1863
             71           1860
             70           1733
        

In [ ]:
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 20}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 30
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df_3 = ensemble.generate(users=val_customers)

print(f"Generated {len(recs_df_3):,} total candidate rows.")
evaluate_candidates(recs_df_3, val_transactions, k=70)
recs_df_3.head()

Generating candidates via ensemble for 68,984 validation customers...
Generated candidate scores for users
Generated 2,716,969 total candidate rows.
Preparing evaluation matrices...
Recall@70: 0.0998


,customer_id,article_id,source
2716968,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0863583001,recency_popularity
1474106,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0707269003,item_similarity
23122,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0881919001,previous_purchases
1474099,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0762846006,item_similarity
1474100,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0915526001,item_similarity


In [ ]:
train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"model_path": als_train_history_file_path, "k": 40})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for train target customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} train target"
    " customers..."
)
recs_df_1 = ensemble.generate(users=train_target_customers)

print(f"Generated {len(recs_df_1):,} total candidate rows.")
evaluate_candidates(recs_df_1, train_target_tx, k=100)

Generating candidates via ensemble for 72,019 train target customers...
Generated als candidates
Generated 1,791,520 total candidate rows.
Preparing evaluation matrices...
Recall@100: 0.0661


0.06614797606783407

In [64]:
recs_df_1.head()

,customer_id,article_id,rank
3779417,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0903994001,1
130070,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0707269007,2
1258705,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0884319005,3
1258704,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0762796013,4
1258703,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0915529001,5


In [21]:
user_counts = train_transactions.groupby("customer_id")["t_dat"].nunique()
    
total_users = len(user_counts)
single_buyers = (user_counts <= 2).sum()
repeat_buyers = (user_counts > 2).sum()

single_buyer_pct = (single_buyers / total_users) * 100

print(f"Total Unique Users: {total_users:,}")
print(f"Less than 3-Purchase Users: {single_buyers:,} ({single_buyer_pct:.2f}%)")
print(f"Repeat Buyers: {repeat_buyers:,} ({100 - single_buyer_pct:.2f}%)")
    

Total Unique Users: 1,356,709
Less than 3-Purchase Users: 628,729 (46.34%)
Repeat Buyers: 727,980 (53.66%)


### New arrivals Candidates

In [29]:
from src.candidate_retrieval.ContentBasedNewArrivals import ContentBasedNewArrivalsCandidates
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "content_based_new_arrivals": (ContentBasedNewArrivalsCandidates, {"lookback_weeks": 12, "k": 20})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(recs_df):,} total candidate rows.")
evaluate_candidates(recs_df, val_transactions, k=20)

Precomputing tiered content-based new arrival candidates...
Tiered content-based precomputation complete!
Generating candidates via ensemble for 68,984 validation customers...
Generated 1,379,362 total candidate rows.
Preparing evaluation matrices...
Recall@20: 0.0041


0.004070425968562042

In [27]:
import pandas as pd

print("Running candidate pool diagnostic...")

# 1. Inspect raw ID formats to catch int vs. string or missing leading zeros
print("\n--- 1. ID Format Inspection ---")
print("Sample val_transactions article_id:", val_transactions['article_id'].head(3).tolist(), type(val_transactions['article_id'].iloc[0]))
print("Sample recs_df article_id:", recs_df['article_id'].head(3).tolist(), type(recs_df['article_id'].iloc[0]))


# 1. Inspect raw ID formats to catch int vs. string or missing leading zeros
print("\n--- 1. ID Format Inspection ---")
print("Sample val_transactions customer_id:", val_transactions['customer_id'].head(3).tolist(), type(val_transactions['customer_id'].iloc[0]))
print("Sample recs_df customer_id:", recs_df['customer_id'].head(3).tolist(), type(recs_df['customer_id'].iloc[0]))

Running candidate pool diagnostic...

--- 1. ID Format Inspection ---
Sample val_transactions article_id: ['0786022008', '0913272003', '0889669006'] <class 'str'>
Sample recs_df article_id: ['0937915001', '0945304001', '0949551002'] <class 'str'>

--- 1. ID Format Inspection ---
Sample val_transactions customer_id: ['000fb6e772c5d0023892065e659963da90b1866035558ec16fca51b0dcfb7e59', '000fb6e772c5d0023892065e659963da90b1866035558ec16fca51b0dcfb7e59', '000fb6e772c5d0023892065e659963da90b1866035558ec16fca51b0dcfb7e59'] <class 'str'>
Sample recs_df customer_id: ['fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20e02ce5d1e58a8f700b', 'fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20e02ce5d1e58a8f700b', 'fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20e02ce5d1e58a8f700b'] <class 'str'>


In [30]:
# Run this to get the exact coverage percentage
val_gt = val_transactions[['customer_id', 'article_id']].copy()
val_gt['article_id'] = val_gt['article_id'].astype(str).str.zfill(10)
val_gt['customer_id'] = val_gt['customer_id'].astype(str)

cand_eval = recs_df.copy()
cand_eval['article_id'] = cand_eval['article_id'].astype(str).str.zfill(10)
cand_eval['customer_id'] = cand_eval['customer_id'].astype(str)

merged_hits = val_gt.merge(
    cand_eval, 
    on=['customer_id', 'article_id'], 
    how='inner'
)

total_gt = len(val_gt)
total_hits = len(merged_hits)
coverage = total_hits / total_gt if total_gt > 0 else 0

print(f"Total validation ground-truth purchases: {total_gt:,}")
print(f"Ground-truth items found in candidate pool: {total_hits:,}")
print(f"Candidate Pool Coverage: {coverage:.4f} ({coverage * 100:.2f}%)")

Total validation ground-truth purchases: 240,311
Ground-truth items found in candidate pool: 986
Candidate Pool Coverage: 0.0041 (0.41%)


### Style Alternatives

In [59]:

val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 12, "k": 20})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
style_recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(style_recs_df):,} total candidate rows.")
evaluate_candidates(style_recs_df, val_transactions, k=20)
style_recs_df.head()

Generating candidates via ensemble for 68,984 validation customers...
Generated 984,146 total candidate rows.
Preparing evaluation matrices...
Recall@20: 0.0085


,customer_id,article_id,rank
111889,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0881891001,1
111879,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0902992001,2
111890,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0895028001,3
111871,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0861121001,4
111872,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0840947002,5


## Final Pipeline and Candidate Generation

### Train target candidate Generation

In [19]:
train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 13, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 7,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 13, "k": 20}),
    "als": (ALSCandidates, {"model_path": als_train_history_file_path, "k": 40}),
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 13, "k": 20})
}
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for train target customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} train target"
    " customers..."
)
train_target_recs_df = ensemble.generate(users=train_target_customers)

print(f"Generated {len(train_target_recs_df):,} total training candidate rows.")
evaluate_candidates(train_target_recs_df, train_target_tx, k=120)

Generating candidates via ensemble for 72,019 train target customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated als candidates
Generated style alternatives candidates
Generated 4,939,981 total training candidate rows.
Preparing evaluation matrices...
Recall@120: 0.1124


0.112412707215924

In [20]:
train_target_recs_df.columns

Index(['customer_id', 'article_id', 'source'], dtype='object')

In [26]:

train_target_recs_df.to_parquet(cleaned_df_folder + "/candidates_train.parquet", index=False)
# Create labels using the training target week transactions
print("Generating labels for candidates")
train_labelled = create_labels(train_target_recs_df, train_target_tx)
# Extract features using the historical transactions
print("Adding features to the candidates")
generator = FeatureGenerator(
    train_transactions_df=train_history_tx,
    articles_df=articles,
    customers_df=customers
)
train_candidate_features = generator.transform(train_labelled)
train_candidate_features.to_parquet(cleaned_df_folder + "/train_candidates_features.parquet", index=False)
# 6. Downsample negatives for training
print("Downsampling candidates")
train_downsampled = downsample_negatives(train_candidate_features, max_negatives_per_user=30)
train_downsampled.to_parquet(cleaned_df_folder + "/train_candidates_downsampled.parquet", index=False)

Generating labels for candidates
Extracting unique purchase pairs from target set...
Found 227,910 unique customer-article purchase pairs in target week.
Total candidates before labeling: 4,939,981
Positive samples captured in candidate pool: 21,412 (0.43%)
Adding features to the candidates
-> Adding item metadata and multi-window popularities...
-> Adding user demographics and profile features...
-> Adding user-item interaction affinity features...
-> Processing candidate retrieval flags and scores...
Feature matrix successfully built. Final shape: (4939981, 28)
Downsampling candidates
Original dataset shape: (4939981, 28)
Original positive count: 21,412
Sampling up to 30 negatives per user...
Downsampled dataset shape: (1985501, 28)
New positive proportion: 0.0108 (1.08%)


### Train Ranker model with train_target candidates

In [6]:
train_downsampled = load_from_parquet("train_candidates_downsampled")

Loading data from Parquet...


In [7]:
# Prepare training structures for LightGBM
train_df = train_downsampled.sort_values("customer_id").reset_index(drop=True)
train_baskets = train_df.groupby("customer_id", sort=False).size().to_numpy()

drop_features = ["customer_id", "article_id", "target"]
feature_cols = [col for col in train_df.columns if col not in drop_features]

X_train = train_df[feature_cols].copy()
y_train = train_df["target"]

# Convert categorical columns
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    
# Save categorical mapping categories for runtime alignment
train_feature_categories = {
    col: X_train[col].cat.categories 
    for col in categorical_cols
}

In [ ]:
print("Training LightGBM ranker model...")
model, categorical_cols = train_ranker(
    X_train,
    y_train,
    train_baskets,
    boosting_type="dart",
    n_estimators=200,
    min_child_samples=75,
    learning_rate=0.05
)

ranker_model_file_path = ROOT_DIR / 'saved_models' / 'ranker_model.pkl'
with open(ranker_model_file_path, 'wb') as f:
    pickle.dump(model, f)

Training LightGBM ranker model...


### Validation Candidates Generation

In [ ]:
# Train als model for val_history
train_and_save_als_model(
    train=train_transactions,
    model_path=als_val_history_file_path,
    lookback_weeks=14,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

In [31]:
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"model_path": als_val_history_file_path, "k": 40}),
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 12, "k": 20})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(recs_df):,} total candidate rows.")
evaluate_candidates(recs_df, val_transactions, k=120)

Generating candidates via ensemble for 68,984 validation customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated als candidates
Generated style alternatives candidates
Generated 4,695,244 total candidate rows.
Preparing evaluation matrices...
Recall@120: 0.1231


0.12308956736624323

In [32]:

recs_df.to_parquet(cleaned_df_folder + "/candidates_val.parquet", index=False)


In [33]:

# Generate features for the Candidates
generator = FeatureGenerator(
    train_transactions_df=train_transactions,
    articles_df=articles,
    customers_df=customers
)

val_candidate_features_df = generator.transform(recs_df)
val_candidate_features_df.to_parquet(cleaned_df_folder + "/val_candidate_features.parquet", index=False)


-> Adding item metadata and multi-window popularities...
-> Adding user demographics and profile features...
-> Adding user-item interaction affinity features...
-> Processing candidate retrieval flags and scores...
Feature matrix successfully built. Final shape: (4695244, 27)


In [10]:
from src.models.rank_model import create_lgbm_recommender
print("Scoring candidates with LightGBM ranker...")
lgbm_recommender_func = create_lgbm_recommender(
    ranker=model,
    candidate_features=val_candidate_features_df,
    categorical_cols=categorical_cols,
    train_feature_categories=train_feature_categories
)
# Generate top 12 recommendations for validation customers
# val_predictions = lgbm_recommender_func(val_customers.tolist(), k=12)

# Prepare ground-truth actual purchases for Week 0
val_gt = val_transactions[['customer_id', 'article_id']]

# Evaluate validation score!
validation_map = evaluate_all(lgbm_recommender_func, val_gt, k=12)

Scoring candidates with LightGBM ranker...
Generating predictions for 68,984 validation customers...
Preparing evaluation matrices...
Computing metrics...


In [11]:

print(f"Validation MAP@12 Score: {validation_map['MAP@12']:.5f}")
print("All Validation Metrics:", validation_map)

Validation MAP@12 Score: 0.02763
All Validation Metrics: {'MAP@12': 0.027628834682045238, 'HitRate@12': 0.12313000115968921, 'NDCG@12': 0.0420292162432148, 'Recall@12': 0.062096010127826916}


## Prediction on Test Dataset

In [25]:
print(len(val_transactions['customer_id'].unique()))

68984


In [8]:
import numpy as np
sample_sub = pd.read_csv("input/raw/sample_submission.csv")
test_customers = sample_sub['customer_id'].unique()
# Combine  training and validation transactions 
train_val_combined = pd.concat([train_transactions, val_transactions], ignore_index=True)

train_val_combined_subset = train_val_combined[train_val_combined['week'] <= 15]
active_customer_set = set(train_val_combined_subset['customer_id'].values)
# Split into active and inactive
print("Splitting test customers...")
test_customers_active = np.array([c for c in test_customers if c in active_customer_set])
test_customers_inactive = np.array([c for c in test_customers if c not in active_customer_set])
print(f"Done! Active: {len(test_customers_active):,}, Inactive: {len(test_customers_inactive):,}")

Splitting test customers...
Done! Active: 594,161, Inactive: 777,819


In [9]:
from src.candidate_retrieval.AlsCandidates import ALSCandidates
print("--- PHASE 2: TEST DATASET PREDICTION  ---")


generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"lookback_weeks": 12, "k": 40}),
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 12, "k": 20})
}

full_ensemble = CandidateGeneratorEnsemble(
    transactions=train_val_combined_subset,  # All data (train + validation combined)
    articles=articles, 
    generators=generators_config
)

# Generate candidates for test customers
print(f"Generating test candidates for {len(test_customers_active):,} test customers...")
test_active_recs_df = full_ensemble.generate(users=test_customers_active)


--- PHASE 2: TEST DATASET PREDICTION  ---
ALS model cache path: C:\Users\yeshw\Documents\Gayathri personal doc\Gayathri\arizona\INFO 556 - Text Retrieval and Web Search\project\hm-fashion-recommendation\saved_models\als_recommender_model_trans_total.pkl
Loading cached ALS model from C:\Users\yeshw\Documents\Gayathri personal doc\Gayathri\arizona\INFO 556 - Text Retrieval and Web Search\project\hm-fashion-recommendation\saved_models\als_recommender_model_trans_total.pkl...
ALS model successfully loaded from cache!
Generating test candidates for 594,161 test customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated als candidates
Generated style alternatives candidates


In [10]:
test_active_recs_df.to_parquet(cleaned_df_folder + "/test_active_candidates.parquet", index=False)



In [11]:
# Candidate Generation for inactive customers

generators_config = {
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20})
}

full_ensemble = CandidateGeneratorEnsemble(
    transactions=train_val_combined_subset,  # All data (train + validation combined)
    articles=articles, 
    generators=generators_config
)

# Generate candidates for test customers
print(f"Generating test candidates for {len(test_customers_inactive):,} test customers...")
test_inactive_recs_df = full_ensemble.generate(users=test_customers_inactive)
test_inactive_recs_df.to_parquet(cleaned_df_folder + "/test_inactive_candidates.parquet", index=False)

Generating test candidates for 777,819 test customers...
Generated recency candidates


In [12]:
print(f"Length of active customer candidates: {len(test_active_recs_df)}")
print(f"Length of inactive customer candidates: {len(test_inactive_recs_df)}")

Length of active customer candidates: 42354409
Length of inactive customer candidates: 15556380


In [13]:
test_candidates_df = pd.concat([test_active_recs_df, test_inactive_recs_df], ignore_index=True)
# Extract features for test candidates
print("Extracting features for test candidates...")
generator = FeatureGenerator(
    train_transactions_df=train_val_combined_subset,
    articles_df=articles,
    customers_df=customers
)

test_candidate_features_df = generator.transform(test_candidates_df)

Extracting features for test candidates...
-> Adding item metadata and multi-window popularities...
-> Adding user demographics and profile features...


MemoryError: Unable to allocate 2.16 GiB for an array with shape (5, 57910789) and data type float64

In [ ]:

# Score and rank them using trained ranker
test_recommender_func = create_lgbm_recommender(
    ranker=model,
    candidate_features=test_candidate_features_df,
    categorical_cols=categorical_cols,
    train_feature_categories=train_feature_categories
)

test_predictions = test_recommender_func(test_customers.tolist(), k=12)

print("Submission file 'submission.csv' created successfully!")

In [ ]:

# Format into Kaggle's required string format (space-separated article IDs)
print("Formatting predictions for Kaggle submission...")
test_predictions['article_id'] = test_predictions['article_id'].apply(lambda x: f"0{x}" if len(str(x)) == 9 else str(x))

submission_df = test_predictions.groupby('customer_id')['article_id'].apply(lambda x: ' '.join(x)).reset_index()
submission_df.rename(columns={'article_id': 'prediction'}, inplace=True)

# Merge back with sample_sub to guarantee every required customer is present
final_submission = sample_sub[['customer_id']].merge(submission_df, on='customer_id', how='left')
# Fill any missing predictions with a blank string or popular default items if needed
final_submission['prediction'] = final_submission['prediction'].fillna('')

# Save to CSV
final_submission.to_csv("submission.csv", index=False)